# Object Oriented Design Metrics — SCIP Semantic Index
<br>

Applies OO Design Metrics (Robert C. Martin's Stable Abstractions Principle) to SCIP Semantic Index modules.

**Abstractness**: ratio of abstract types to total types per module. `isAbstract = true` on `SemanticCodeIndexInternalType` nodes.

**Instability**: ratio of outgoing to total (outgoing + incoming) module-level `DEPENDS_ON` relationships.

**Main Sequence**: ideal balance line where `Abstractness + Instability = 1`.

**Zone of Pain**: concrete + stable = hard to change without breaking dependents.  
**Zone of Uselessness**: abstract + unstable = over-designed, unlikely to be used.

Visibility metrics are not applicable for SCIP modules and are omitted.

### References
- [Analyze java package metrics in a graph database](https://joht.github.io/johtizen/data/2023/04/21/java-package-metrics-analysis.html)
- [Calculate metrics](https://101.jqassistant.org/calculate-metrics/index.html)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plot
from matplotlib.colors import LinearSegmentedColormap
from neo4j import GraphDatabase

In [ ]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell 
# before starting jupyter notebook to provide the password for the user "neo4j".
# It is not recommended to hardcode the password into jupyter notebook for security reasons.

driver = GraphDatabase.driver(uri="bolt://localhost:7687", auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD")))
driver.verify_connectivity()

In [ ]:
def get_cypher_query_from_file(cypher_file_name: str) -> str:
    with open(cypher_file_name) as file:
        return ' '.join(file.readlines())


def query_cypher_to_data_frame(filename: str, limit: int = -1) -> pd.DataFrame:
    """
    Execute the Cypher query of the given file and return the result as a DataFrame.

    Args:
        filename: Path to the file containing the Cypher query.
        limit: Optional row limit appended to the query. Default -1 = no limit.
    """
    cypher_query = get_cypher_query_from_file(filename)
    if limit > 0:
        cypher_query = f"{cypher_query}\nLIMIT {limit}"
    records, summary, keys = driver.execute_query(cypher_query)
    return pd.DataFrame([r.values() for r in records], columns=keys)


def query_first_non_empty_cypher_to_data_frame(*filenames: str, limit: int = -1) -> pd.DataFrame:
    """
    Execute the Cypher queries of the given files and return the first non-empty result.

    If all queries return empty results, the last (empty) result is returned.
    The second file is typically a SET query that writes and returns data when the
    GET query (first file) returns nothing (property not yet set).
    """
    result = pd.DataFrame()
    for filename in filenames:
        result = query_cypher_to_data_frame(filename, limit)
        if not result.empty:
            return result
    return result

In [ ]:
#The following cell uses the build-in %html "magic" to override the CSS style for tables to a much smaller size.
#This is especially needed for PDF export of tables with multiple columns.

In [ ]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [ ]:
# Colormap for Main Sequence scatter: green (near main sequence) → red (far away)
MAIN_SEQUENCE_COLORMAP = LinearSegmentedColormap.from_list(
    "main_sequence", ["green", "gold", "orangered", "red"], N=256
)

pd.set_option('display.max_colwidth', 300)

## 1 — Incoming Dependencies

Incoming dependencies are also denoted as "Fan-in", "Afferent Coupling", or "in-degree".
These are modules that depend on the listed module. High incoming = widely used, hard to change safely.

The `incomingDependencies` property is set during SCIP enrichment in the scip-index-import domain.

#### Table 1a — Top 20 SCIP modules with the most incoming dependencies

In [ ]:
query_cypher_to_data_frame(
    "../queries/object-oriented-design-metrics/Get_Incoming_SCIP_Module_Dependencies.cypher",
    limit=20,
)

## 2 — Outgoing Dependencies

Outgoing dependencies are also denoted as "Fan-out", "Efferent Coupling", or "out-degree".
These are modules that this module depends on. High outgoing = broad consumer; sensitive to upstream changes.

#### Table 2a — Top 20 SCIP modules with the most outgoing dependencies

In [ ]:
query_cypher_to_data_frame(
    "../queries/object-oriented-design-metrics/Get_Outgoing_SCIP_Module_Dependencies.cypher",
    limit=20,
)

## 3 — Instability

$$Instability = \frac{Outgoing\:Dependencies}{Outgoing\:Dependencies + Incoming\:Dependencies}$$

*Instability* measures how many changes in other modules could force this module to change.
- `0` = fully stable (only incoming, no outgoing) — risky to change, others depend on it.
- `1` = fully unstable (only outgoing, no incoming) — free to change, nothing depends on it.

#### Table 3a — Top 20 SCIP modules with the lowest Instability

Sets the `instability` property on module nodes if not already done.

In [ ]:
query_first_non_empty_cypher_to_data_frame(
    "../queries/object-oriented-design-metrics/Get_Instability_for_SCIP.cypher",
    "../queries/object-oriented-design-metrics/Calculate_and_set_Instability_for_SCIP.cypher",
    limit=20,
)

## 4 — Abstractness

$$Abstractness = \frac{Abstract\:Types}{Total\:Types}$$

*Abstractness* = ratio of abstract types to total types per module.
A `SemanticCodeIndexInternalType` is abstract when `isAbstract = true`.

- `0` = fully concrete — easy to use, but hard to extend without modification.
- `1` = fully abstract — all types are abstract/interface; flexible but requires concrete implementations.

#### Table 4a — Top 20 SCIP modules with the lowest Abstractness

Sets `numberOfAbstractTypes`, `numberOfTypes`, and `abstractness` on module nodes if not already done.

In [ ]:
# Count_and_set_abstract_types_for_SCIP must run before Calculate_and_set_Abstractness_for_SCIP
query_cypher_to_data_frame(
    "../queries/object-oriented-design-metrics/Count_and_set_abstract_types_for_SCIP.cypher"
)

query_first_non_empty_cypher_to_data_frame(
    "../queries/object-oriented-design-metrics/Get_Abstractness_for_SCIP.cypher",
    "../queries/object-oriented-design-metrics/Calculate_and_set_Abstractness_for_SCIP.cypher",
    limit=20,
)

## 5 — Distance from Main Sequence

$$Distance = |Abstractness + Instability - 1|$$

The **Main Sequence** is the ideal diagonal where `A + I = 1`. Distance measures how far a module deviates.

- **Zone of Pain** (`A ≈ 0, I ≈ 0`): concrete + stable = risky, hard to change.
- **Zone of Uselessness** (`A ≈ 1, I ≈ 1`): abstract + unstable = over-designed, likely unused.

#### Table 5a — Top 20 SCIP modules furthest from Main Sequence

In [ ]:
main_sequence_data = query_cypher_to_data_frame(
    "../queries/object-oriented-design-metrics/Calculate_distance_between_abstractness_and_instability_for_SCIP.cypher",
    limit=20,
)
main_sequence_data.head(20)

#### Chart 5b — Main Sequence scatter plot

- X-axis: Abstractness (0 = concrete, 1 = abstract)
- Y-axis: Instability (0 = stable, 1 = unstable)
- Point size: number of types (larger = more types in module)
- Color: distance from Main Sequence (green = near, red = far)
- Green dashed diagonal = Main Sequence ideal line

In [ ]:
all_main_sequence_data = query_cypher_to_data_frame(
    "../queries/object-oriented-design-metrics/Calculate_distance_between_abstractness_and_instability_for_SCIP.cypher"
)

if not all_main_sequence_data.empty and {'abstractness', 'instability', 'elementsCount', 'distance'}.issubset(all_main_sequence_data.columns):
    marker_scales = all_main_sequence_data['elementsCount'].clip(lower=2, upper=300) * 0.7

    figure, axis = plot.subplots(figsize=(10, 8))
    scatter = axis.scatter(
        all_main_sequence_data['abstractness'],
        all_main_sequence_data['instability'],
        s=marker_scales,
        c=all_main_sequence_data['distance'],
        cmap=MAIN_SEQUENCE_COLORMAP,
        alpha=0.5,
    )
    axis.plot([0, 1], [1, 0], color='lightgreen', linestyle='dashed', label='Main Sequence')
    figure.colorbar(scatter, ax=axis, label='Distance from Main Sequence')
    axis.set_title('SCIP Modules — Abstractness vs. Instability (Main Sequence)')
    axis.set_xlabel('Abstractness')
    axis.set_ylabel('Instability')
    axis.set_xlim(-0.05, 1.05)
    axis.set_ylim(-0.05, 1.05)
    axis.legend(loc='upper right')
    plot.tight_layout()
    plot.show()
else:
    print("No data available for Main Sequence scatter plot.")

#### Table 5c — Modules in Zone of Pain (concrete + stable)

In [ ]:
if not all_main_sequence_data.empty:
    zone_of_pain = all_main_sequence_data.query('abstractness <= 0.3 and instability <= 0.3').head(20)
    print(f"Zone of Pain modules: {len(zone_of_pain)}")
    zone_of_pain

#### Table 5d — Modules in Zone of Uselessness (abstract + unstable)

In [ ]:
if not all_main_sequence_data.empty:
    zone_of_uselessness = all_main_sequence_data.query('abstractness >= 0.7 and instability >= 0.7').head(20)
    print(f"Zone of Uselessness modules: {len(zone_of_uselessness)}")
    zone_of_uselessness